# 05. Contrastive Learning — SimCLR 1-cycle Experiment
# 05. Contrastive Learning — SimCLR 1회 실험

**Goal / 목표**
- Run one cycle of SimCLR Contrastive Learning (Phase 0) without labels
- 라벨 없이 SimCLR Contrastive Learning (Phase 0) 1회 순환을 실행한다
- Verify that the model learns meaningful embeddings from unlabeled images
- 라벨 없는 이미지에서 의미있는 embedding을 학습하는지 확인한다
- This is a prototype — no Swing Architecture, no Optuna
- 이 노트북은 프로토타입으로 Swing Architecture 및 Optuna 없이 실행한다

**SimCLR 핵심 아이디어 / Key Idea**
```
같은 이미지의 두 augmentation → Positive Pair → 가깝게
Different augmentations of the same image → Positive Pair → pull together

다른 이미지 → Negative Pair → 멀게
Different images → Negative Pair → push apart
```

**Execution Order / 실행 순서**
1. Import libraries / 라이브러리 임포트
2. Path & settings / 경로 및 설정
3. Contrastive dataset / Contrastive 데이터셋 구성
4. Build model (Backbone + Projection Head) / 모델 구성
5. InfoNCE Loss / InfoNCE Loss 정의
6. Training loop / 학습 루프
7. Verify results / 결과 확인

---
## 1. Import Libraries / 라이브러리 임포트

In [ ]:
import time
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models
from pathlib import Path

# 재현성을 위한 시드 고정 / Fix seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 디바이스 설정 / Device setup
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)

print('Libraries loaded / 라이브러리 로드 완료')
print(f'PyTorch version / 버전: {torch.__version__}')
print(f'Device / 디바이스: {device}')

---
## 2. Path & Settings / 경로 및 설정

> config.yaml 없이 직접 경로와 값을 지정한다.
> Paths and values are set directly without config.yaml.

In [ ]:
ROOT_DIR    = Path('../..').resolve()                # CMYK_MAIN/
LABELED_DIR = ROOT_DIR / 'data_set' / 'labeled'     # 라벨링 폴더 / Labeled folder
MODEL_DIR   = ROOT_DIR / 'data_set' / 'models'      # 모델 저장 폴더 / Model save folder
REPORT_DIR  = ROOT_DIR / 'data_set' / 'reports'     # 리포트 폴더 / Report folder

CHANNELS      = ['Y', 'M', 'C', 'K']
NUM_LEVELS    = 6            # Level 0 ~ 5
IMAGE_SIZE    = 128          # R1 기준 크기 / R1 standard size
BACKBONE_NAME = 'efficientnet_b0'  # 'efficientnet_b0' | 'resnet50'

# SimCLR 하이퍼파라미터 / SimCLR hyperparameters
EPOCHS         = 10      # 프로토타입용 축약 / Shortened for prototype
BATCH_SIZE     = 16      # 배치 크기 (클수록 Negative Pair 많아짐) / Larger = more negative pairs
LEARNING_RATE  = 1e-3    # 초기 학습률 / Initial learning rate
TEMPERATURE    = 0.1     # InfoNCE τ — sharpness 제어 / Controls similarity sharpness
PROJECTION_DIM = 128     # Projection Head 출력 차원 / Projection Head output dimension
TARGET_CHANNEL = 'C'     # 테스트할 채널 / Channel to test — Y / M / C / K

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Backbone / 백본: {BACKBONE_NAME}')
print(f'Target channel / 대상 채널: {TARGET_CHANNEL}')
print(f'Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | τ: {TEMPERATURE} | Proj dim: {PROJECTION_DIM}')
print(f'Labeled dir / 라벨 경로: {LABELED_DIR}')

# 데이터 수 확인 / Check data counts
total = 0
for lv in range(NUM_LEVELS):
    folder = LABELED_DIR / TARGET_CHANNEL / str(lv)
    count  = len(list(folder.glob('*'))) if folder.exists() else 0
    total += count
    print(f'  [{TARGET_CHANNEL}] Level {lv}: {count}개')
print(f'  [{TARGET_CHANNEL}] Total / 전체: {total}개')

---
## 3. Contrastive Dataset / Contrastive 데이터셋 구성

- Loads images WITHOUT labels — labels are not needed for Phase 0
- 라벨 없이 이미지만 로드한다 — Phase 0에서는 라벨이 필요 없다
- Returns two different augmented views of the same image as a Positive Pair
- 같은 이미지의 두 가지 증강을 Positive Pair로 반환한다

```
image → augment → view1 ┐
                         ├→ Positive Pair
image → augment → view2 ┘
```

In [ ]:
class ContrastiveDataset(Dataset):
    """
    Phase 0 Contrastive Learning 전용 Dataset.
    라벨 없이 이미지만 로드하며, 같은 이미지의 두 augmentation을 Positive Pair로 반환한다.

    Phase 0 Contrastive Learning Dataset.
    Loads images without labels and returns two augmented views as a Positive Pair.

    폴더 구조 / Folder structure:
        data_set/labeled/{channel}/{level}/*.png  (레벨 무시 / level ignored)
    """

    def __init__(self, channel: str):
        self.image_paths = []
        self.exts        = {'.png', '.jpg', '.jpeg', '.tiff', '.tif'}

        # 모든 레벨의 이미지 수집 (라벨 무시) / Collect all images (ignore labels)
        channel_dir = LABELED_DIR / channel
        for level in range(NUM_LEVELS):
            level_dir = channel_dir / str(level)
            if not level_dir.exists():
                continue
            for img_path in sorted(level_dir.glob('*')):
                if img_path.suffix.lower() in self.exts:
                    self.image_paths.append(img_path)

        print(f'[{channel}] Contrastive dataset / 데이터셋: {len(self.image_paths)}개 이미지 / images')

    def __len__(self) -> int:
        return len(self.image_paths)

    def _augment(self, image: np.ndarray) -> torch.Tensor:
        """
        Contrastive Learning 전용 강한 증강을 적용한다.
        Applies strong augmentation for Contrastive Learning.
        """
        # 수평 뒤집기 / Horizontal flip
        if random.random() > 0.5:
            image = cv2.flip(image, 1)

        # 랜덤 크롭 후 리사이즈 / Random crop and resize
        if random.random() > 0.5:
            h, w  = image.shape[:2]
            scale = random.uniform(0.6, 1.0)
            ch, cw = int(h * scale), int(w * scale)
            y0 = random.randint(0, h - ch)
            x0 = random.randint(0, w - cw)
            image = image[y0:y0+ch, x0:x0+cw]
            image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))

        # 밝기 변동 / Brightness jitter
        if random.random() > 0.5:
            value = random.uniform(-0.2, 0.2)
            image = np.clip(image + value, 0, 1)

        # 대비 변동 / Contrast jitter
        if random.random() > 0.5:
            factor = random.uniform(0.8, 1.2)
            image  = np.clip((image - 0.5) * factor + 0.5, 0, 1)

        # 가우시안 블러 / Gaussian blur
        if random.random() > 0.5:
            k     = random.choice([3, 5])
            image = cv2.GaussianBlur(
                (image * 255).astype(np.uint8), (k, k), 0
            ).astype(np.float32) / 255.0

        # (H, W, 3) → (3, H, W) 텐서 변환 / Convert to tensor
        return torch.tensor(image).permute(2, 0, 1).float()

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]

        # 이미지 로드 / Load image
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f'Image not found / 이미지 없음: {img_path}')

        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        image = image.astype(np.float32) / 255.0  # [0,1] 정규화 / Normalize

        # Positive Pair: 같은 이미지의 두 augmentation
        # Positive Pair: two different augmentations of the same image
        view1 = self._augment(image.copy())
        view2 = self._augment(image.copy())

        return view1, view2


# 데이터셋 및 DataLoader 구성 / Build dataset and DataLoader
dataset = ContrastiveDataset(TARGET_CHANNEL)

if len(dataset) == 0:
    print(f'[WARN] No data / 데이터 없음: {LABELED_DIR / TARGET_CHANNEL}')
else:
    loader = DataLoader(
        dataset,
        batch_size=min(BATCH_SIZE, len(dataset)),
        shuffle=True,
        drop_last=True,   # 마지막 불완전 배치 제거 (InfoNCE에서 중요) / Drop last batch (important for InfoNCE)
        num_workers=0,
    )
    print(f'DataLoader / 배치 수: {len(loader)} batches')

    # Positive Pair 확인 / Verify positive pair
    v1, v2 = next(iter(loader))
    print(f'View1 shape: {v1.shape} | View2 shape: {v2.shape}')

---
## 4. Build Model (Backbone + Projection Head) / 모델 구성

- Phase 0 전용 구조 / Phase 0 exclusive structure
- Backbone → Projection Head (256 → 128)
- Projection Head는 Phase 2 진입 시 제거하고 Classifier Head로 교체한다
- Projection Head is removed and replaced with Classifier Head upon entering Phase 2

In [ ]:
class ProjectionHead(nn.Module):
    """
    Phase 0 Contrastive Learning 전용 Projection Head.
    Phase 0 Contrastive Learning exclusive Projection Head.

    Backbone 출력 → Linear(256) → BN → ReLU → Linear(128)
    Backbone output → Linear(256) → BN → ReLU → Linear(128)
    """

    def __init__(self, in_dim: int, hidden_dim: int = 256, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),   # 배치 정규화 / Batch normalization
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
            # Phase 2 진입 시 이 Head 전체를 제거하고 ClassifierHead로 교체
            # Remove this entire Head and replace with ClassifierHead upon Phase 2
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)  # (B, 128) projection vector 반환 / Returns projection vector


class SimCLRModel(nn.Module):
    """Backbone + ProjectionHead 전체 모델 / Full SimCLR model."""

    def __init__(self, backbone, feature_dim: int, proj_dim: int = 128):
        super().__init__()
        self.backbone = backbone
        self.head     = ProjectionHead(feature_dim, out_dim=proj_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)   # 특징 추출 / Extract features
        return self.head(features)    # Projection vector 반환 / Return projection vector


# Backbone 로드 / Load backbone
if BACKBONE_NAME == 'efficientnet_b0':
    from torchvision.models import EfficientNet_B0_Weights
    backbone    = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    feature_dim = backbone.classifier[1].in_features  # 1280
    backbone.classifier = nn.Identity()               # Head 제거 / Remove head

elif BACKBONE_NAME == 'resnet50':
    from torchvision.models import ResNet50_Weights
    backbone    = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    feature_dim = backbone.fc.in_features              # 2048
    backbone.fc = nn.Identity()                        # Head 제거 / Remove head

else:
    raise ValueError(f'Unsupported backbone / 지원하지 않는 backbone: {BACKBONE_NAME}')

model = SimCLRModel(backbone, feature_dim, proj_dim=PROJECTION_DIM).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model built / 모델 구성 완료: {BACKBONE_NAME}')
print(f'Feature dim / 특징 차원: {feature_dim} → Projection dim: {PROJECTION_DIM}')
print(f'Total params / 전체 파라미터: {total_params:,}')
print(f'Trainable params / 학습 파라미터: {trainable_params:,}')

---
## 5. InfoNCE Loss

- SimCLR 기반 InfoNCE Loss
- Positive Pair (같은 이미지의 두 augmentation) 를 가깝게, Negative Pair 를 멀게 학습
- Pulls Positive Pairs (two augmentations of the same image) together, pushes Negative Pairs apart
- τ (temperature) 가 작을수록 분포가 sharp해짐 / Smaller τ = sharper distribution

In [ ]:
class InfoNCELoss(nn.Module):
    """
    SimCLR 기반 InfoNCE Loss.
    SimCLR-based InfoNCE Loss.

    Args:
        temperature: τ — 유사도 분포의 sharpness 제어 / Controls sharpness of similarity distribution
    """

    def __init__(self, temperature: float = 0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z1: (B, D) — view1 projection vectors
            z2: (B, D) — view2 projection vectors

        Returns:
            scalar loss
        """
        B      = z1.size(0)
        device = z1.device

        # L2 정규화 — 코사인 유사도 계산을 위해 단위 벡터로 변환
        # L2 normalization — convert to unit vectors for cosine similarity
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # (2B, D) 결합 — view1과 view2를 하나의 배치로 합침
        # Concatenate (2B, D) — merge view1 and view2 into a single batch
        z = torch.cat([z1, z2], dim=0)

        # 유사도 행렬 (2B, 2B) — temperature로 스케일링
        # Similarity matrix (2B, 2B) — scaled by temperature
        sim = torch.matmul(z, z.T) / self.temperature

        # 자기 자신과의 유사도 제거 (대각선 마스킹)
        # Mask out self-similarity (diagonal masking)
        mask = torch.eye(2*B, dtype=torch.bool, device=device)
        sim.masked_fill_(mask, -1e9)

        # Positive pair 인덱스: i ↔ i+B
        # view1[i]의 Positive는 view2[i], view2[i]의 Positive는 view1[i]
        # Positive of view1[i] is view2[i], Positive of view2[i] is view1[i]
        labels = torch.cat([
            torch.arange(B, 2*B, device=device),
            torch.arange(0, B,   device=device),
        ])

        # CrossEntropyLoss로 Positive pair를 최대화, Negative pair를 최소화
        # Maximize positive pair similarity, minimize negative pairs via CrossEntropyLoss
        loss = F.cross_entropy(sim, labels)
        return loss


# Loss 테스트 / Loss test
criterion = InfoNCELoss(temperature=TEMPERATURE)
z_test1   = torch.randn(4, PROJECTION_DIM)
z_test2   = torch.randn(4, PROJECTION_DIM)
test_loss = criterion(z_test1, z_test2)
print(f'InfoNCE Loss test / 테스트: {test_loss.item():.4f} (τ={TEMPERATURE})')
print(f'Expected range / 예상 범위: ~{torch.log(torch.tensor(8.0)).item():.2f} (log(2B) = log({2*4}))')

---
## 6. Training Loop / 학습 루프

- Train for EPOCHS iterations
- EPOCHS 동안 학습한다
- Save backbone weights after training for Phase 2 use
- 학습 완료 후 Phase 2 사용을 위해 Backbone weights를 저장한다

In [ ]:
if len(dataset) == 0:
    print('[WARN] Skipping training / 학습 건너뜀 — 데이터 없음 / No data')
else:
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    history = []

    print('=' * 55)
    print(f'  SimCLR Training Start / 학습 시작 — Channel: [{TARGET_CHANNEL}]')
    print(f'  τ={TEMPERATURE} | Proj dim={PROJECTION_DIM} | Epochs={EPOCHS}')
    print('=' * 55)
    print(f'  {"Epoch":<8} {"Loss":<14} {"LR":<14} Elapsed')
    print('-' * 55)

    for epoch in range(1, EPOCHS + 1):
        t0         = time.time()
        total_loss = 0.0

        model.train()  # train 모드 — BN, Dropout 활성화 / Activate BN, Dropout

        for view1, view2 in loader:
            view1, view2 = view1.to(device), view2.to(device)

            # Positive Pair forward pass
            z1   = model(view1)              # (B, 128) projection vector
            z2   = model(view2)              # (B, 128) projection vector
            loss = criterion(z1, z2)         # InfoNCE Loss 계산 / Compute InfoNCE Loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / max(len(loader), 1)
        scheduler.step()
        elapsed = time.time() - t0
        lr      = optimizer.param_groups[0]['lr']

        record = {
            'epoch':   epoch,
            'loss':    round(avg_loss, 6),
            'lr':      round(lr, 8),
            'elapsed': round(elapsed, 2),
        }
        history.append(record)

        print(f'  {epoch:<8} {avg_loss:<14.4f} {lr:<14.2e} {elapsed:.1f}s')

    print('-' * 55)
    print(f'  Training done / 학습 완료')
    print(f'  First loss / 초기 loss: {history[0]["loss"]:.4f}')
    print(f'  Last loss  / 최종 loss: {history[-1]["loss"]:.4f}')

    # Backbone weights 저장 (Phase 2 에서 사용)
    # Save backbone weights (for use in Phase 2)
    backbone_path = MODEL_DIR / f'phase0_backbone_{TARGET_CHANNEL}.pt'
    torch.save(model.state_dict(), backbone_path)
    print(f'  Backbone saved / 저장: {backbone_path}')
    print('=' * 55)

---
## 7. Verify Results / 결과 확인

- Check if loss decreased over training
- 학습 중 loss가 감소했는지 확인한다
- Verify saved backbone weights can be loaded
- 저장된 Backbone weights가 로드 가능한지 확인한다

> Loss 감소 = Positive Pair가 가까워지고 Negative Pair가 멀어지고 있음을 의미한다.
> Loss decrease = Positive Pairs are getting closer and Negative Pairs are being pushed apart.

In [ ]:
if len(dataset) == 0:
    print('[WARN] No data — skipping / 데이터 없음 — 건너뜀')
else:
    # 학습 이력 출력 / Print training history
    print('학습 이력 / Training History:')
    print(f'  {"Epoch":<8} {"Loss"}')
    print('-' * 20)
    for r in history:
        print(f'  {r["epoch"]:<8} {r["loss"]:.4f}')

    # Loss 감소 확인 / Check loss decrease
    first_loss = history[0]['loss']
    last_loss  = history[-1]['loss']
    decreased  = last_loss < first_loss

    print(f'\nLoss 변화 / Loss change: {first_loss:.4f} → {last_loss:.4f}')
    if decreased:
        print('  [PASS] Loss decreased — embeddings are being learned / Loss 감소 — embedding 학습 중')
    else:
        print('  [WARN] Loss did not decrease — check batch size or data count')
        print('         데이터 수나 배치 크기를 확인하세요')

    # 저장된 Backbone 로드 확인 / Verify saved backbone can be loaded
    print('\nBackbone 로드 확인 / Backbone load verification:')
    test_model = SimCLRModel(backbone, feature_dim, proj_dim=PROJECTION_DIM).to(device)
    test_model.load_state_dict(torch.load(backbone_path, map_location=device))
    test_model.eval()

    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
    with torch.no_grad():
        out = test_model(dummy)

    if out.shape == (1, PROJECTION_DIM):
        print(f'  [PASS] Backbone loaded and forward pass OK / 로드 성공')
        print(f'         Output shape: {tuple(out.shape)} (expected (1, {PROJECTION_DIM}))')
    else:
        print(f'  [FAIL] Output shape mismatch: {tuple(out.shape)}')

    print(f'\n다음 단계 / Next step:')
    print(f'  03_training.ipynb 에서 phase0_backbone_{TARGET_CHANNEL}.pt 를 로드하여 Phase 2 학습을 진행한다.')
    print(f'  Load phase0_backbone_{TARGET_CHANNEL}.pt in 03_training.ipynb to start Phase 2 training.')